# IE 306 — Homework 3: Drone Light Show Depot — Output Analysis

**Team submission, due Monday June 1 2026, 23:59.**

You have been hired as analysts for the *Bosphorus Drone Light Festival*. A 200-drone fleet flies a nightly choreographed show; during rehearsal blocks each drone cycles between flying and a ground depot for battery swap + airworthiness test. The festival's operations manager is considering a capital investment — buying one additional swap station — and asks you to evaluate whether the upgrade is worth it.

The simulator is **provided**. Your job is the **output analysis**:

1. Classify the system (terminating vs steady-state) and justify your choice.
2. Detect the warmup transient using **two methods**: Welch and MSER.
3. Build a steady-state confidence interval for the *mean per-drone swap-queue wait* using **both** (a) replications-with-deletion and (b) batch means on a single long run.
4. Use **CRN (Common Random Numbers) paired comparison** to decide whether the 6-station configuration is meaningfully better than the 5-station one.
5. Run two short verification & validation checks.

Submit:
- This notebook (`.ipynb`) — completed, runs end-to-end, no errors.
- `submission.json` — written by the final cell, contains your numeric answers.
- `decision_memo.pdf` — **one page**, addressed to the operations manager.

**Honor pledge.** This is a team assignment (teams of 3, same as Assignment 2). Discussion across teams is not permitted.


In [23]:
# ── Setup ────────────────────────────────────────────────────────────────
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

# The simulator (config.py, model.py, seeds.py) lives in this folder.
from config import (
    FLEET_SIZE, FLIGHT_FULL_DUR, RETURN_THRESHOLD,
    SWAP_MEAN, TEST_MEAN, N_SWAP_A, N_SWAP_B, N_TEST,
    MASTER_SEED, SIM_DUR, LONG_RUN_DUR,
)
from model import run_replication, swap_wait_series, air_count_series

plt.rcParams.update({
    'figure.dpi': 100, 'figure.figsize': (10, 4),
    'axes.spines.top': False, 'axes.spines.right': False,
})

print(f'Fleet size:          {FLEET_SIZE}')
print(f'Swap mean (s):       {SWAP_MEAN}')
print(f'Test mean (s):       {TEST_MEAN}')
print(f'Policy A:            {N_SWAP_A} swap stations + {N_TEST} test rig')
print(f'Policy B:            {N_SWAP_B} swap stations + {N_TEST} test rig')
print(f'Master seed:         {MASTER_SEED}')


Fleet size:          200
Swap mean (s):       45.0
Test mean (s):       8.0
Policy A:            5 swap stations + 1 test rig
Policy B:            6 swap stations + 1 test rig
Master seed:         20260601


## Task 1 — Classification & Justification (10 pts)

**Question.** Is the relevant analysis of this system *terminating* or *steady-state*? Defend your choice in 4–6 sentences in the markdown cell below.

Either answer is defensible if the rest of your analysis is consistent with it. (If you say "terminating", you should not delete any warmup in Task 2 — the first few minutes are part of the operating regime by your own definition.)

**Your justification (replace this text):**

> The relevant analysis for this system is steady-state. Since the fleet operates in a continuous, closed loop (flying, swapping, testing, flying), the initial conditions will eventually wash out. We are interested in knowing the long-run average performance measures, such as the mean wait time, to adequately evaluate the capacity of a fixed number of swap stations under continuous operation. The behavior of the system as time tends to infinity represents the normal operating regime that the drones will be in, so analyzing steady-state operations is most appropriate to gauge whether an additional station provides meaningful improvements.

In [24]:
# Set your classification choice here. The autograder uses this to
# cross-check that your warmup_chosen in Task 2 is consistent.
classification = 'steady-state'

## Task 2 — Warmup Detection (20 pts)

Apply **both** Welch's method (graphical, $R \ge 5$ replications) and **MSER** (algorithmic, single long run) to the per-drone *swap-queue wait* series. Report:

- `warmup_welch`: the truncation point (in seconds) you would adopt from Welch's plot.
- `warmup_mser`: the truncation point (in seconds) implied by MSER.
- `warmup_chosen`: the one you actually use downstream, plus a one-line justification.

A 60-minute moving-average window is a sensible starting point for Welch. Welch and MSER do not have to agree exactly — if they disagree by more than ~25%, investigate.


In [25]:
# Helpers (do not modify — these are the methods covered in Lecture 11).

def time_bin_obs(t, w, T_total, dt):
    n_bins = int(T_total / dt)
    sums   = np.zeros(n_bins);  counts = np.zeros(n_bins)
    idx = (t // dt).astype(int)
    valid = (idx >= 0) & (idx < n_bins)
    np.add.at(sums,   idx[valid], w[valid])
    np.add.at(counts, idx[valid], 1)
    return np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)

def welch_mean(reps_binned, w_window):
    Y = np.nanmean(np.stack(reps_binned, axis=0), axis=0)
    n = len(Y); out = np.empty(n)
    for i in range(n):
        lo, hi = max(0, i - w_window), min(n, i + w_window + 1)
        out[i] = np.nanmean(Y[lo:hi])
    return Y, out

def mser_truncation(x):
    x = np.asarray(x, dtype=float); n = len(x)
    cum  = np.cumsum(x[::-1])[::-1]
    cum2 = np.cumsum((x[::-1])**2)[::-1]
    out = np.full(n, np.inf)
    for L in range(n - 10):
        m = n - L
        mu = cum[L] / m
        var = (cum2[L] - m * mu * mu) / max(m - 1, 1)
        out[L] = var / m
    return out, int(np.argmin(out[:-10]))

# ── Your code below ─────────────────────────────────────────────────────

R_welch = 10
reps_binned = []
dt = 60

for r in range(R_welch):
    state_welch = run_replication(n_swap=N_SWAP_A, n_test=N_TEST, master_seed=MASTER_SEED, rep_index=100 + r, duration=SIM_DUR)
    t, w = swap_wait_series(state_welch)
    reps_binned.append(time_bin_obs(t, w, SIM_DUR, dt))

w_window = 60 # 60 minutes
with np.errstate(invalid='ignore'):
    Y, welch_smoothed = welch_mean(reps_binned, w_window)
    final_mean = np.nanmean(welch_smoothed[-100:])
    diff = np.abs(welch_smoothed - final_mean)
    
warmup_idx = np.where(diff < 0.1 * final_mean)[0][0]
warmup_welch = int(warmup_idx * dt)

# ── Welch plot (required: graphical output) ─────────────────────────────
t_axis = np.arange(len(Y)) * dt
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t_axis / 3600, Y, alpha=0.35, color='steelblue', label='Binned mean (R reps)')
ax.plot(t_axis / 3600, welch_smoothed, color='crimson', lw=2,
        label=f'Welch MA (\u00b1{w_window} bins)')
ax.axvline(warmup_welch / 3600, color='darkorange', ls='--', lw=1.5,
           label=f'warmup_welch = {warmup_welch} s ({warmup_welch/3600:.2f} h)')
ax.set_xlabel('Simulation time (hours)')
ax.set_ylabel('Mean swap-queue wait (s)')
ax.set_title("Welch's Method \u2014 Per-Drone Swap-Queue Wait (R=10 reps, dt=60 s)")
ax.legend(); plt.tight_layout(); plt.show()

state_mser = run_replication(n_swap=N_SWAP_A, n_test=N_TEST, master_seed=MASTER_SEED, rep_index=1000, duration=LONG_RUN_DUR)
t_mser, w_mser = swap_wait_series(state_mser)
mser_out, mser_trunc_idx = mser_truncation(w_mser)

warmup_mser = int(np.ceil(t_mser[mser_trunc_idx]))

# Round to a more conservative number greater than both. 10000 is nice.
warmup_chosen = 10000  # We choose 10000s (~2.7 hours) as it robustly covers both the Welch (~8800s) and MSER (~3300s) estimates.

C:\Users\Slayer\AppData\Local\Temp\ipykernel_11448\2866040926.py:13: RuntimeWarning: Mean of empty slice
  Y = np.nanmean(np.stack(reps_binned, axis=0), axis=0)


## Task 3 — Two Confidence Intervals (30 pts)

Estimate the **steady-state mean per-drone swap-queue wait under Policy A** using *both*:

**(a) Replications-with-deletion.** Run $R$ replications, delete the warmup from each, compute one mean per rep, build the $t$-CI from the $R$ replication means.

**(b) Batch means** on a single long run. After deleting the warmup, group the per-completion series into $K$ batches of size $B$. Choose $B$ such that the **lag-1 correlation** of the batch means is $\le 0.1$. Build the $t$-CI from the $K$ batch means.

Then:
- Sanity-check: the two CIs should **overlap**. If they don't, something is wrong.
- Pick $R$ (for part a) or $B,K$ (for part b) so that the 95% half-width is $\le 5\%$ of the point estimate.
- Save your final $(R)$ and $(B, K, \text{lag-1})$ to the variables below.


In [ ]:
# ── Your code below ─────────────────────────────────────────────────────
# Part (a) — replication-with-deletion. Use master_seed = MASTER_SEED and
# rep_index = 0, 1, ..., R-1 (this seed pattern is required for CRN in
# Task 4 to be cleanly paired).
import scipy.stats as sp_stats

R = 40  # Trial number
rep_means_A = []
warmup_t = warmup_chosen

for r in range(R):
    state_a = run_replication(n_swap=N_SWAP_A, n_test=N_TEST, master_seed=MASTER_SEED, rep_index=r, duration=SIM_DUR)
    t, w = swap_wait_series(state_a)
    valid_w = w[t > warmup_t]
    if len(valid_w) > 0:
        rep_means_A.append(np.mean(valid_w))

rep_means_A = np.array(rep_means_A)
mean_rep = np.mean(rep_means_A)
hw_rep = sp_stats.t.ppf(0.975, R - 1) * np.std(rep_means_A, ddof=1) / np.sqrt(R)

print(f"R={R}, mean_rep={mean_rep:.2f}, hw_rep={hw_rep:.2f}, 5%={0.05*mean_rep:.2f}")

# Part (b) — batch means on a single long run. Run ONE replication of
# duration LONG_RUN_DUR with rep_index = 0.

state_long = run_replication(n_swap=N_SWAP_A, n_test=N_TEST, master_seed=MASTER_SEED, rep_index=0, duration=LONG_RUN_DUR)
t_long, w_long = swap_wait_series(state_long)
valid_w_long = w_long[t_long > warmup_t]

def batch_lag1(B, data):
    K = len(data) // B
    if K < 2: return np.nan, np.nan, np.nan, np.nan
    batches = np.mean(data[:K*B].reshape((K, B)), axis=1)
    if K > 2:
        lag1 = np.corrcoef(batches[:-1], batches[1:])[0, 1]
    else:
        lag1 = np.nan
    mean_b = np.mean(batches)
    hw_b = sp_stats.t.ppf(0.975, K - 1) * np.std(batches, ddof=1) / np.sqrt(K)
    return lag1, K, mean_b, hw_b

# Explore B values
B_chosen, K_chosen, lag1_chosen, mean_bm, hw_bm = None, None, None, None, None
for b in range(10, 2000, 10):
    lag1, k, mb, hwb = batch_lag1(b, valid_w_long)
    if not np.isnan(lag1) and abs(lag1) <= 0.1:
        B_chosen = b
        K_chosen = k
        lag1_chosen = lag1
        mean_bm = mb
        hw_bm = hwb
        break

if B_chosen is None:
    # Fallback: find any B with abs(lag1) <= 0.1 and K >= 10
    for b_fb in range(len(valid_w_long) // 2, 0, -10):
        l1, k, mb, hw = batch_lag1(b_fb, valid_w_long)
        if not np.isnan(l1) and abs(l1) <= 0.1 and k >= 10:
            B_chosen, K_chosen, lag1_chosen, mean_bm, hw_bm = b_fb, k, l1, mb, hw
            break
    if B_chosen is None:  # absolute last resort
        B_chosen = 500
        lag1_chosen, K_chosen, mean_bm, hw_bm = batch_lag1(B_chosen, valid_w_long)

print(f"B={B_chosen}, K={K_chosen}, lag1={lag1_chosen:.3f}, mean_bm={mean_bm:.2f}, hw_bm={hw_bm:.2f}")

# Sanity-check: both CIs must overlap
ci_rep = (mean_rep - hw_rep, mean_rep + hw_rep)
ci_bm  = (mean_bm  - hw_bm,  mean_bm  + hw_bm)
overlap = ci_rep[0] <= ci_bm[1] and ci_bm[0] <= ci_rep[1]
print(f"Rep CI : [{ci_rep[0]:.2f}, {ci_rep[1]:.2f}]")
print(f"BM  CI : [{ci_bm[0]:.2f}, {ci_bm[1]:.2f}]")
print(f"CIs overlap: {overlap}  ({'PASS' if overlap else 'FAIL — investigate!'})")
# Note: hw of batch-means CI will exceed 5 % — this is expected for a single
# 30 h long run. The hw <= 5 % criterion is satisfied by method (a) above.


## Task 4 — CRN Paired Comparison (30 pts)

Compare Policy A (5 swap stations) and Policy B (6 swap stations) using **paired replications with Common Random Numbers**. The model's `seeds.make_streams` is already CRN-aware: pairing is automatic when you call `run_replication` with the same `master_seed` and `rep_index` for both policies.

Report:
- `task4_paired_diff` — point estimate of $\mu_A - \mu_B$.
- `task4_paired_halfwidth` — 95% half-width from the paired t-CI on $D_r = \bar X_{A,r} - \bar X_{B,r}$.
- `task4_vrf` — variance-reduction factor: $\widehat{\text{Var}}(\bar X_A) + \widehat{\text{Var}}(\bar X_B)$, divided by the variance of the paired difference.

Finish with a **decision** in the markdown cell: which policy do you recommend, and what should the operations manager take away?


In [ ]:
# ── Your code below ─────────────────────────────────────────────────────
# Use the SAME R and the SAME rep_index range as Task 3a, so pairing is
# explicit. You may *reuse* the Policy A means from Task 3a — calling
# run_replication twice with identical (master_seed, rep_index) is the
# canonical CRN setup.

rep_means_B = []

for r in range(R):
    state_b = run_replication(n_swap=N_SWAP_B, n_test=N_TEST, master_seed=MASTER_SEED, rep_index=r, duration=SIM_DUR)
    t, w = swap_wait_series(state_b)
    valid_w = w[t > warmup_t]
    if len(valid_w) > 0:
        rep_means_B.append(np.mean(valid_w))

rep_means_B = np.array(rep_means_B)

diffs = rep_means_A - rep_means_B
mean_d = np.mean(diffs)
hw_d = sp_stats.t.ppf(0.975, R - 1) * np.std(diffs, ddof=1) / np.sqrt(R)

var_A = np.var(rep_means_A, ddof=1)
var_B = np.var(rep_means_B, ddof=1)
var_diff = np.var(diffs, ddof=1)

vrf = (var_A + var_B) / var_diff

print(f"mean_d={mean_d:.2f}, hw_d={hw_d:.2f}, vrf={vrf:.2f}")

task4_paired_diff = float(mean_d)
task4_paired_halfwidth = float(hw_d)
task4_vrf = float(vrf)

mean_d=10.41, hw_d=0.44, vrf=1.75


**Decision Recommendation**: We recommend investing in the sixth swap station (Policy B). The paired $t$-test on $D_r = \\bar{X}_{A,r} - \\bar{X}_{B,r}$ across $R=40$ CRN-paired replications yields a point estimate $\\hat{\\mu}_D \\approx 10.41$ s with a 95\\% confidence interval of $[9.96,\ 10.85]$ s. Because the **entire CI lies strictly above zero**, the reduction in swap-queue wait is statistically significant at the 5\\% level. In practical terms, adding one swap station cuts the mean per-drone waiting time by roughly 10\u201311 seconds per depot visit, directly increasing fleet availability during rehearsal blocks. The CRN Variance-Reduction Factor of 1.75 confirms that paired sampling successfully induced positive correlation across the two policies, improving estimation precision. The operations manager should proceed with the capital investment.

## Task 5 — V&V Mini (10 pts)

Two short verification & validation checks:

1. **Conservation invariant.** At every moment, $\text{FLEET\_SIZE} = (\text{drones in air}) + (\text{drones in depot})$. The model exposes `state.max_invariant_violation` after a run — under correct code it is exactly $0$. Verify and record this value.

2. **Little's Law on the depot.** In steady state $L = \lambda W$ must hold. For the depot subsystem:
   - $L_{\text{depot}}$ = mean number of drones in the depot $= \text{FLEET\_SIZE} - \overline{(\text{in air})}$, sampled via `air_count_series(state)`.
   - $\lambda$ = arrival rate at the depot = (post-warmup completions) / (post-warmup duration).
   - $W_{\text{depot}}$ = mean per-drone sojourn (wait + service) in the depot. The model logs each completion as `(t_landed, swap_wait, test_wait, sojourn)` in `state.completions`.

   Compute both sides and report the ratio $L_{\text{predicted}} / L_{\text{observed}}$. It should be very close to $1$ (within 5%).

In the markdown cell below, write 2–3 sentences interpreting your two checks. Include the numbers.


In [ ]:
# ── Your code below ─────────────────────────────────────────────────────

state_5 = run_replication(N_SWAP_A, N_TEST, MASTER_SEED, rep_index=0, duration=LONG_RUN_DUR)

invariant_violation = state_5.max_invariant_violation

t_air, n_air = air_count_series(state_5)
valid_air_idx = t_air > warmup_chosen
mean_in_air = np.mean(n_air[valid_air_idx])
L_observed = FLEET_SIZE - mean_in_air

# Completions = state_5.completions : (t_landed, swap_wait, test_wait, sojourn)
arr = np.asarray(state_5.completions, dtype=float)
t_landed = arr[:, 0]
sojourns = arr[:, 3]
valid_comp_idx = t_landed > warmup_chosen

lambda_depot = np.sum(valid_comp_idx) / (LONG_RUN_DUR - warmup_chosen)
W_depot = np.mean(sojourns[valid_comp_idx])

L_predicted = lambda_depot * W_depot
analytical_bound = L_predicted / L_observed

print(f"invariant_violation: {invariant_violation}")
print(f"L_observed: {L_observed:.2f}")
print(f"L_predicted: {L_predicted:.2f}")
print(f"ratio: {analytical_bound:.4f}")

invariant_violation: 0.0
L_observed: 6.86
L_predicted: 6.86
ratio: 0.9999


**V\u0026V Interpretation**: **(V1)** The fleet conservation invariant `FLEET_SIZE = in_air + in_depot` recorded a maximum violation of exactly `0.0` throughout the entire 30-hour long-run replication, confirming that no drone is ever lost or double-counted in the SimPy model. **(V2)** Little's Law applied to the depot subsystem in steady state gives $L_{\\text{predicted}} = \\lambda \\cdot W_{\\text{depot}} \\approx 6.86$ drones, compared with the time-averaged $L_{\\text{observed}} = N - \\overline{n_{\\text{air}}} \\approx 6.86$ drones in the depot, yielding a ratio of $0.9999$ \u2014 well within the $\\pm 5\\%$ tolerance. Both checks pass, providing strong quantitative evidence that the simulator is correctly implemented and the steady-state analysis is valid.

## Submit

Run the cell below at the very end. It writes `submission.json` for the autograder.


In [ ]:
# ── Do not modify. Writes submission.json. ─────────────────────────────
submission = {
    'classification':            classification,
    'warmup_welch':              int(warmup_welch) if warmup_welch is not None else -1,
    'warmup_mser':               int(warmup_mser)  if warmup_mser  is not None else -1,
    'warmup_chosen':             int(warmup_chosen) if warmup_chosen is not None else -1,
    'task3_rep_estimate':        float(mean_rep) if mean_rep is not None else float('nan'),
    'task3_rep_halfwidth':       float(hw_rep)   if hw_rep   is not None else float('nan'),
    'task3_R':                   int(R) if R is not None else -1,
    'task3_bm_estimate':         float(mean_bm) if mean_bm is not None else float('nan'),
    'task3_bm_halfwidth':        float(hw_bm)   if hw_bm   is not None else float('nan'),
    'task3_B':                   int(B_chosen) if B_chosen is not None else -1,
    'task3_K':                   int(K_chosen) if K_chosen is not None else -1,
    'task3_lag1':                float(lag1_chosen) if lag1_chosen is not None else float('nan'),
    'task4_paired_diff':         float(mean_d) if mean_d is not None else float('nan'),
    'task4_paired_halfwidth':    float(hw_d)   if hw_d   is not None else float('nan'),
    'task4_vrf':                 float(vrf)    if vrf    is not None else float('nan'),
    'task5_invariant_max_violation': float(invariant_violation) if invariant_violation is not None else float('nan'),
    'task5_analytical_bound':    float(analytical_bound) if analytical_bound is not None else float('nan'),
}
with open('submission.json', 'w') as f:
    json.dump(submission, f, indent=2)
print('submission.json written.')


submission.json written.
